In [ ]:
import pandas as pd

df = pd.read_csv("train.csv")
print(df.info())
print(df.head())

<class 'pandas.DataFrame'>
RangeIndex: 614 entries, 0 to 613
Data columns (total 13 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   Loan_ID            614 non-null    str    
 1   Gender             601 non-null    str    
 2   Married            611 non-null    str    
 3   Dependents         599 non-null    str    
 4   Education          614 non-null    str    
 5   Self_Employed      582 non-null    str    
 6   ApplicantIncome    614 non-null    int64  
 7   CoapplicantIncome  614 non-null    float64
 8   LoanAmount         592 non-null    float64
 9   Loan_Amount_Term   600 non-null    float64
 10  Credit_History     564 non-null    float64
 11  Property_Area      614 non-null    str    
 12  Loan_Status        614 non-null    str    
dtypes: float64(4), int64(1), str(8)
memory usage: 62.5 KB
None
    Loan_ID Gender Married Dependents     Education Self_Employed  \
0  LP001002   Male      No          0      Graduate  

In [ ]:
print(df.isnull().sum())

Loan_ID               0
Gender               13
Married               3
Dependents           15
Education             0
Self_Employed        32
ApplicantIncome       0
CoapplicantIncome     0
LoanAmount           22
Loan_Amount_Term     14
Credit_History       50
Property_Area         0
Loan_Status           0
dtype: int64


In [ ]:
df = df.drop(columns= ['Loan_ID'])

In [ ]:
print(df['Credit_History'].value_counts())

Credit_History
1.0    475
0.0     89
Name: count, dtype: int64


In [ ]:
from sklearn.base import TransformerMixin, BaseEstimator # forgot the base estimator
from helpers import binary_mapper

class Binary_Mapper(BaseEstimator, TransformerMixin):
    def fit(self, X, y= None): #wrote small none
        return self
    def transform(self, X): #forgot the arguments/parameters here hehe ;p
        X = X.copy()
        X['Gender'] = binary_mapper(X['Gender'], {'male':1, 'female': 0}, 'Gender') # totally forgot how to write this line haha
        X['Married'] = binary_mapper(X['Married'], {'yes': 1, 'no': 0}, 'Married')
        X['Education'] = binary_mapper(X['Education'],{'graduate': 1, 'not graduate': 0}, 'Education')
        X['Self_Employed'] = binary_mapper(X['Self_Employed'],{'yes': 1, 'no': 0}, 'Self_Employed')
        return X
        

In [ ]:
df['Loan_Status'] = df['Loan_Status'].map({'Y': 1, 'N': 0})

In [ ]:
class FeatureEngineer(BaseEstimator, TransformerMixin):
    def fit(self, X, y = None):
        self.median_ = X['LoanAmount'].median()
        return self
    def transform(self, X):
        X = X.copy()
        X['Income_to_Loan'] = X['ApplicantIncome']/X['LoanAmount'].fillna(self.median_)
        return X


In [ ]:
categorical_columns = ['Dependents', 'Property_Area']
numeric_columns = ['ApplicantIncome', 'CoapplicantIncome', 'LoanAmount', 'Loan_Amount_Term', 'Income_to_Loan']
binary_columns = ['Gender', 'Married', 'Education', 'Self_Employed', 'Credit_History']


In [ ]:
from sklearn.model_selection import train_test_split # forgot this module too huhu

X = df.drop(columns=['Loan_Status'])
y = df["Loan_Status"]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state= 42, stratify= y) # forgot the test size and stratify writing here.. 


In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import OneHotEncoder

binary_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy= 'most_frequent')),
])

categorical_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('encoder', OneHotEncoder(handle_unknown= 'ignore', drop = 'first'))
])

numeric_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy= 'median')),
    ('scaler', StandardScaler())
])

#forgot all the modules and pipeline syntax - like wth is this ;-;

In [ ]:
from sklearn.compose import ColumnTransformer   # Forgot about it entirely ;-;

preprocessor = ColumnTransformer([
    ("bin", binary_pipeline, binary_columns), 
    ("cat", categorical_pipeline, categorical_columns),
    ("num", numeric_pipeline, numeric_columns)
])

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn.model_selection import cross_val_score # Yeah forgot all these too ;-;

models = {
    "Logistic Regression": LogisticRegression(max_iter=1000, random_state= 42, class_weight= 'balanced'),
    "Decision Tree": DecisionTreeClassifier(random_state=42, class_weight='balanced'),
    "RandomForestClassifier": RandomForestClassifier(random_state=42, class_weight= 'balanced')
}

In [ ]:
for name, model in models.items():
    pipe = Pipeline([
        ('Binary_Mapper', Binary_Mapper()),
        ('Feature_Engineer', FeatureEngineer()),
        ('preprocessor', preprocessor),
        ('model', model)
    ])
    scores = cross_val_score(pipe, X_train, y_train, cv = 5, scoring = 'f1')
    print(f"{name}: {scores.mean():.4f} +/- {scores.std():.4f}")

ValueError: 
All the 5 fits failed.
It is very likely that your model is misconfigured.
You can try to debug the error by setting error_score='raise'.

Below are more details about the failures:
--------------------------------------------------------------------------------
5 fits failed with the following error:
Traceback (most recent call last):
  File "/home/bibek/.local/lib/python3.14/site-packages/sklearn/model_selection/_validation.py", line 833, in _fit_and_score
    estimator.fit(X_train, y_train, **fit_params)
    ~~~~~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/home/bibek/.local/lib/python3.14/site-packages/sklearn/base.py", line 1336, in wrapper
    return fit_method(estimator, *args, **kwargs)
  File "/home/bibek/.local/lib/python3.14/site-packages/sklearn/pipeline.py", line 613, in fit
    Xt = self._fit(X, y, routed_params, raw_params=params)
  File "/home/bibek/.local/lib/python3.14/site-packages/sklearn/pipeline.py", line 547, in _fit
    X, fitted_transformer = 

In [ ]:
fitted_pipelines = {}

for name, model in models.items():
    pipe = Pipeline([
        ('Binary_Mapper', Binary_Mapper()),
        ('FeatureEngineer', FeatureEngineer()),
        ('preprocessor', preprocessor),
        ('model', model)
    ])

    pipe.fit(X_train, y_train)
    fitted_pipelines[name] = pipe

    train_acc = accuracy_score(y_train, pipe.predict(X_train))
    test_acc = accuracy_score(y_test, pipe.predict(X_test))
    print(f"{name}: Test: {test_acc:.4f} | Train:  {train_acc:.4f}")

Logistic Regression: Test: 0.8537 | Train:  0.7536
Decision Tree: Test: 0.7154 | Train:  1.0000
RandomForestClassifier: Test: 0.8455 | Train:  1.0000


In [ ]:
row = pd.DataFrame([{
    "Gender": "Male", "Married": "no", "Dependents": "0", "Education": "Graduate",
    "Self_Employed": "No", "ApplicantIncome": 5000, "CoapplicantIncome": 1500,
    "LoanAmount": 128, "Loan_Amount_Term": 360, "Credit_History": 1.0,
    "Property_Area": "Urban",
}])
fitted_pipelines['Logistic Regression'].predict_proba(row)

array([[0.34880609, 0.65119391]])